In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import gc
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from pathlib import Path

# 1. CONFIGURATION
class CFG:
    SR = 32000
    DURATION = 5 
    CHUNK_SIZE = SR * DURATION
    # Competition Data Path (Updated for BirdCLEF 2026)
    INPUT_DIR = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes/")
    MODEL_PATH = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/rare-amphibian-oversample/3/bird_model_best.pth"
    
    
    NUM_CLASSES = 234 
    THRESHOLD = 0.00

for dirname, _, filenames in os.walk('/kaggle/input/competitions/birdclef-2026/test_soundscapes/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import timm
import torch.nn as nn

# REFINED MODEL CLASS
class BirdModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', num_classes=234):
        super().__init__()
        # 1. Create model with 1 input channel instead of 3
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=1)
        
        # 2. Based on your error message, your trained weights use "fc" 
        # instead of "backbone.classifier". Let's map it:
        self.fc = nn.Linear(self.backbone.classifier.in_features, num_classes)
        
        # 3. Disable the default classifier so it doesn't conflict
        self.backbone.classifier = nn.Identity()
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x) # Use the "fc" layer as in your training
        return x

In [ ]:
# 3. INITIALIZATION
device = torch.device(
    "cuda" if torch.cuda.is_available() else  # Fixed: is_available()
    "mps" if torch.backends.mps.is_available() else 
    "cpu"
)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Using device: {device}")

# model = BirdModel()
model = BirdModel(model_name='efficientnet_b3', num_classes=234)

state_dict = torch.load(CFG.MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print(f"Using device: {device}")

In [ ]:
# 4. LOAD SUBMISSION FORMAT + CREATE SPECIES MAPPING

# Load the official sample submission to get the 234 required species columns
sample_sub_path = Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)
target_species = sample_sub.columns[1:].tolist()

In [ ]:
# 5. the data is here:
data_root = Path("/kaggle/input/competitions/birdclef-2026")

# Load taxonomy to get all 234 species codes
taxonomy_path = Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
taxonomy_df = pd.read_csv(taxonomy_path)

# In BirdCLEF, the standard is alphabetical order of species codes (labels)
# Use 'species_code' or the primary label column used during your training
# The competition uses 'primary_label' as the unique identifier
all_species_codes = sorted(taxonomy_df['primary_label'].unique()) 

# Create the mapping that matches your model's 234 output neurons
species_mapping = {species: i for i, species in enumerate(all_species_codes)}

print(f"✅ Mapping created for {len(species_mapping)} species.")
# Verification check:
if len(species_mapping) != 234:
    print(f"⚠️ Warning: Found {len(species_mapping)} species, expected 234!")

In [ ]:
# 6. INFERENCE LOGIC

def get_spectrogram(audio_chunk):
    # Matches the 1-channel logic your model expects
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=CFG.SR, n_mels=128, fmax=16000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB


def get_raw_probabilities(file_path):
    """
    Handles audio loading and generates raw model predictions for every 5s chunk.
    """
    y, _ = librosa.load(file_path, sr=CFG.SR)
    raw_probs = []
    
    for i in range(0, len(y), CFG.CHUNK_SIZE):
        chunk = y[i:i + CFG.CHUNK_SIZE]
        if len(chunk) < CFG.CHUNK_SIZE:
            chunk = np.pad(chunk, (0, CFG.CHUNK_SIZE - len(chunk)))
        
        spec = get_spectrogram(chunk)
        # Add batch and channel dimensions: [1, 1, H, W]
        tensor_spec = torch.from_numpy(spec)[None, None, ...].float().to(device)
        
        with torch.no_grad():
            output = model(tensor_spec)
            probs = torch.sigmoid(output).cpu().numpy()[0]
        
        raw_probs.append(probs)
    
    del y
    return np.array(raw_probs)

In [ ]:
# Cell 7: Temporal Formatting

def aggregate_and_format(raw_probs, file_id, target_species, species_mapping):
    """
    Applies improved temporal smoothing (max + weighted avg)
    and formats results for Kaggle submission.
    """
    smoothed_probs = np.copy(raw_probs).astype(np.float32)
    num_chunks = len(raw_probs)
    
    # --- TEMPORAL AGGREGATION (Improved) ---
    for i in range(num_chunks):
        if num_chunks <= 1:
            break

        if i == 0:
            avg = 0.7 * raw_probs[i] + 0.3 * raw_probs[i + 1]
            smoothed_probs[i] = np.maximum(raw_probs[i], avg)

        elif i == num_chunks - 1:
            avg = 0.7 * raw_probs[i] + 0.3 * raw_probs[i - 1]
            smoothed_probs[i] = np.maximum(raw_probs[i], avg)

        else:
            avg = (
                0.5 * raw_probs[i] +
                0.25 * raw_probs[i - 1] +
                0.25 * raw_probs[i + 1]
            )
            smoothed_probs[i] = np.maximum(raw_probs[i], avg)

    # --- FORMATTING ---
    file_results = []

    for i in range(num_chunks):
        row_id = f"{file_id}_{(i + 1) * 5}"
        pred_dict = {"row_id": row_id}
        
        chunk_probs = smoothed_probs[i]

        # ✅ Vectorized thresholding (clean + faster)
        chunk_probs = np.where(chunk_probs > CFG.THRESHOLD, chunk_probs, 0.0)

        for species in target_species:
            if species in species_mapping:
                pred_dict[species] = chunk_probs[species_mapping[species]]
            else:
                pred_dict[species] = 0.0
                
        file_results.append(pred_dict)
        
    return file_results

In [ ]:
# Cell 8

def predict_file(file_path):
    """
    Main entry point used by the loop.
    """
    # 1. Get raw numbers from the model
    raw_probs = get_raw_probabilities(file_path)
    
    # 2. Smooth and format
    results = aggregate_and_format(
        raw_probs, 
        file_path.stem, 
        target_species, 
        species_mapping
    )
    
    gc.collect()
    return results

In [ ]:
# 9. MAIN SUBMISSION LOOP
# Check for hidden test files; fallback to train files for the 'Commit' run
test_dir = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes")
test_files = list(test_dir.glob("*.ogg"))

if len(test_files) == 0:
    print("No test files found. Running on subset of training data for verification.")
    test_dir = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")
    test_files = list(test_dir.glob("*.ogg"))[:3]

all_preds = []
for file in tqdm(test_files):
    all_preds.extend(predict_file(file))

# 10. GENERATE SUBMISSION FILE
submission_df = pd.DataFrame(all_preds)
submission_df.to_csv("submission.csv", index=False)
print(f"Successfully generated submission.csv with {len(submission_df)} rows.")